# Financial Risk Report Generation

## Set up

In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
import os
import pandas as pd
import yfinance as yf
from bs4 import BeautifulSoup
import requests
from neo4j import GraphDatabase
import numpy as np
import json
from io import StringIO
import time

import warnings

warnings.filterwarnings("ignore")

In [2]:
load_dotenv()

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
NEO_PASSWORD = os.environ.get("NEO_PASSWORD")
NEO_USERNAME = os.environ.get("NEO_USERNAME")
NEO_URL = os.environ.get("NEO_URL")
NEO_DATABASE = os.environ.get("NEO_DATABASE")

client = OpenAI()

In [3]:
# graph initialization
driver = GraphDatabase.driver(
    NEO_URL,
    auth=(NEO_USERNAME, NEO_PASSWORD),
    database=NEO_DATABASE
)

## Functions

In [ ]:
def get_prompts(company,company_cik,year,sample_report="",quantitative_data="",peer_quantitative_data="",peer_company_name="",parent_subsidiary_triples="",issuer_table="",owner_table="",questions="",subgraph_triples="",report_v1="",report_v2="",report_v3=""):
        
    BASE = f"""You are an expert in finance and you are writing a financial credit risk report for the company {company} (with CIK {company_cik}) and year {year}. 
    You will generate a section called key rating drivers, which consists on a list of the main reasons behind the assigned credit risk rating, usually including strengths and weaknesses.
    Try to focus mainly in negative impacts and risks and avoid giving to many positive key drivers.
    Follow the style and structure of rating commentaries, with a main title for the key driver and a detailed description of it. 
    Note that the report should focus in 3 types of factors:
        - F1-Financial Profile: Quantitative indicators of company's financial strength, profitability, financial structure and financial flexibility.
        - F2-Business Profile: Internal strategic and organizational characteristics, including competitive positioning, managerial decisions, ownership and subsidiary structure, and governance quality.
        - F3-Operating Environment: External macroeconomic, sectoral, regulatory and other external conditions shaping the firm risk context.
    """

    V_ALL = f"""Generate the financial report focusing exclusively on the most relevant factors from these reports: {report_v1}, {report_v2}, {report_v3}. 
        
        Feel free to combine and relate key drivers from different reports.
        Avoid excesive summarization or omitting key information and data from report's most relevant sections.
        
        Use a beautiful readable format, organizing the drivers in sections and including a short introduction and a conclusion.     
        """
    
    PROMPTS = {
        "v0":BASE,    
        
        "v1":BASE + f"""You will now focus only in key rating drivers related to the Financial Profile (F1)
                    The information provided in the generated report should come from the data you can find in this table: 
                    {quantitative_data}                    
                    and the answers to this questions:    
                    {questions}
                    
                    For peer comparison, use the data available from {peer_company_name}, which is in the same sector as {company}:
                    {peer_quantitative_data}
                    
                    Do not reference the questions, table or data in the report text. Just use it for your insights.""",
                    
        "v2":BASE + f"""You will now focus only in key rating drivers related to the Business Profile (F2)
                    The information provided in the generated report should come from the data you can find in this tables: 
                    The first table is the issuer transactions table:
                    {issuer_table}
                    The second table is the owner transactions table:
                    {owner_table}
                    (use only if not empty)
                    
                    Additionally, consider (if available) the company group structure (parents and subsidiaries) from this Knowledge Subgraph:
                    {parent_subsidiary_triples}
                    
                    Generate your response based on the answers to this questions: 
                    {questions}
                    
                    Do not reference the questions, table or data in the report text. Just use it for your insights.""",
                    
        "v3":BASE + f"""You will now focus only in key rating drivers related to the Operating Environment (F3)
        The information provided in the generated report should come exclusively from the data you can find in this Knowledge Subgraph: 
        {subgraph_triples} 
        and the answers to this questions: 
        {questions}
        
        Do not reference the questions or data in the report text. Just use it for your insights.""",
        
        "v_all":BASE +  V_ALL,
        
        "v_all_with_sample": BASE +  V_ALL + f"""This is an example of a list with 3 key rating drivers (one of each type): 
        {sample_report}""",
    }

    return PROMPTS

In [ ]:
def generate_report(company,company_cik,year,prompt_type="v0",self_correct=True,**kwargs):

    first_prompt = get_prompts(company,company_cik,year,**kwargs)[prompt_type]
    
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "user", "content": first_prompt}
        ]
    )

    report_first = response.choices[0].message.content
    
    if not self_correct:
        return report_first
    
    messages = [
        {"role": "user", "content": first_prompt},         
        {"role": "assistant", "content": report_first},    
        {"role": "user", "content": """Now correct and improve the previous report ignoring the list of questions from the first petition. Preserve the list of key rating drivers format, without adding extra titles or sections such as "additional observations". 
                                        - Eliminate any hallucinations, inaccuracies, or irrelevant key rating drivers. 
                                        - Make sure all key risks are included and detailed, and add any additional observations or insights from the data that were missed in the first response. 
                                        - Remove any point that seem ambiguous or not insightful."""}
    ]

    response_corrected = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages
    )

    report_self_corrected = response_corrected.choices[0].message.content

    return report_first, report_self_corrected

In [63]:
def get_fitch_metrics_timeseries(ticker_symbol, report_year=2026, n_years=3):
    ticker = yf.Ticker(ticker_symbol)

    bs = ticker.balance_sheet.dropna(axis=1, how="all")      # Balance Sheet
    cf = ticker.cashflow.dropna(axis=1, how="all")          # Cash Flow
    is_ = ticker.financials.dropna(axis=1, how="all")       # Income Statement

    last_report_year = report_year - n_years 

    # get all the columns until the last report year (it may be in different positions depending on the company data)

    cols = [col for col in is_.columns if last_report_year <= col.year <= report_year]
    rows = []

    for col in cols:
        # --- Raw metrics ---
        revenue = is_.loc["Total Revenue", col] if "Total Revenue" in is_.index else None
        ebitda = is_.loc["EBITDA", col] if "EBITDA" in is_.index else None
        interest_exp = is_.loc["Interest Expense", col] if "Interest Expense" in is_.index else None

        total_debt = bs.loc["Total Debt", col] if "Total Debt" in bs.index else None
        cash = bs.loc["Cash And Cash Equivalents", col] if "Cash And Cash Equivalents" in bs.index else None
        st_debt = bs.loc["Current Debt", col] if "Current Debt" in bs.index else None
        net_debt = bs.loc["Net Debt", col] if "Net Debt" in bs.index else None

        cfo = cf.loc["Operating Cash Flow", col] if "Operating Cash Flow" in cf.index else None
        capex = cf.loc["Capital Expenditure", col] if "Capital Expenditure" in cf.index else None      
        fcf = cf.loc["Free Cash Flow", col] if "Free Cash Flow" in cf.index else None 
        
        ffo = cfo + cf.loc["Change In Working Capital",col]
        
        interest_paid = abs(cf.loc["Interest Paid Supplemental Data",col]) if "Interest Paid Supplemental Data" in cf.index else 0
        preferred_divs = abs(cf.loc["Cash Dividends Paid",col]) if "Cash Dividends Paid" in cf.index else 0

        # --- Derived Fitch ratios ---
        row = {
            "Period": col.strftime("%Y-%m-%d") if hasattr(col, "strftime") else str(col),
            
            "EBITDA": ebitda,
            "Cash": cash,
            "CapEx":capex,

            # Margins
            "EBITDA Margin": (ebitda / revenue) if (ebitda and revenue) else None,

            # Cash flow levels
            "FFO": ffo,
            "CFO": cfo,
            "FCF": fcf,

            # Coverage
            "FFO Interest Coverage": (ffo + interest_paid + preferred_divs) / (interest_paid + preferred_divs),

            # Leverage
            "FFO leverage": (total_debt / (ffo + interest_paid + preferred_divs)) if (total_debt and ffo) else None,
            "EBITDA leverage": (total_debt/ebitda),

            # Free cash flow ratio
            "Free Cash Flow Ratio": (fcf / total_debt) if (fcf and total_debt) else None,

            # Liquidity
            "Cash/ST Debt": (cash / st_debt) if (cash and st_debt) else None,
        }

        rows.append(row)

    df = pd.DataFrame(rows).set_index("Period").dropna(axis=0, how="all") # drop rows in which all data is missing
    return df

# run this for a fast test
# ticker_symbol = "MSFT"

ticker_symbol = "ALK"

result_df = get_fitch_metrics_timeseries(ticker_symbol,report_year=2022,n_years=4)
result_df

,EBITDA,Cash,CapEx,EBITDA Margin,FFO,CFO,FCF,FFO Interest Coverage,FFO leverage,EBITDA leverage,Free Cash Flow Ratio,Cash/ST Debt
Period,,,,,,,,,,,,
2022-12-31,5.880000e+08,338000000.0,-1.671000e+09,0.060958,1.825000e+09,1.418000e+09,-253000000.0,26.704225,1.993671,6.428571,-0.066931,1.224638
2021-12-31,1.140000e+09,470000000.0,-2.920000e+08,0.184585,1.148000e+09,1.030000e+09,738000000.0,11.532110,3.250597,3.584211,0.180617,1.284153


In [68]:
# peer analysis functions
def get_peers_from_neo(cik,limit=10):
    """Extract <limit> companies from the same sector as the company with cik=<cik>"""
    query = f"""
    MATCH p=(n:Company{{cik:"{cik}"}})-[:BELONGS_TO_INDUSTRY_OF]->()<-[:BELONGS_TO_INDUSTRY_OF]-(m:Company) RETURN m.ticker, m.name LIMIT {limit};"""

    with driver.session() as session:
        result = session.run(query, cik=cik)
        return result.data()

def get_company_size(ticker_symbol, year=2025):
    # no funciona seleccionar el año por limites de la api
    ticker = yf.Ticker(ticker_symbol)
    bs = ticker.balance_sheet
    #total_assets = bs.loc["Total Assets", bs.columns.year == year]
    size = bs.loc["Total Assets"].iloc[0] if "Total Assets" in bs.index else 0
    
    #print(ticker_symbol, size)
    #return total_assets.iloc[0] if not total_assets.empty else 0
    return size
    
def get_peer_data(company_cik,company_ticker, year=2025):
    # we first extract a sample of copmanies from the same sector
    candidates = get_peers_from_neo(company_cik)
    # get the asset size, the selected peer will be the company with the closest asset size
    min_diff = np.inf
    size_diff = 0
    peer_ticker = ""
    peer_name = ""
    
    company_size = get_company_size(company_ticker, year=year)
    
    for candidate in candidates:
        size_diff = abs(company_size-get_company_size(candidate["m.ticker"]))
        # print(size_diff)
        if size_diff < min_diff:
            peer_ticker = candidate["m.ticker"]
            peer_name = candidate["m.name"]
            min_diff = size_diff
    
    # we select the company with smallest size difference
    # print("selected company",peer_name,peer_ticker)
    return peer_name, peer_ticker


company_ticker = "MRK"
company_cik = "0000310158"

peer_name, peer_ticker = get_peer_data(company_cik,company_ticker)
peer_quantitative_data = get_fitch_metrics_timeseries(peer_ticker, report_year=2022)
peer_quantitative_data
# ticker = yf.Ticker(peer_ticker)

# bs = ticker.balance_sheet      # Balance Sheet
# cf = ticker.cashflow           # Cash Flow
# is_ = ticker.financials        # Income Statement

# is_

,EBITDA,Cash,CapEx,EBITDA Margin,FFO,CFO,FCF,FFO Interest Coverage,FFO leverage,EBITDA leverage,Free Cash Flow Ratio,Cash/ST Debt
Period,,,,,,,,,,,,
2022-12-31,2.417400e+10,9.201000e+09,-695000000.0,0.416405,2.484900e+10,2.494300e+10,2.424800e+10,2.973866,1.690021,2.617316,0.383240,2.224613
2021-12-31,2.393300e+10,9.746000e+09,-787000000.0,0.425877,2.154600e+10,2.277700e+10,2.199000e+10,2.799549,2.287777,3.204111,0.286761,0.779992


In [ ]:
def get_issuer_owner_tables(company_cik,year=None):
    companyIssuerUrl = r"https://www.sec.gov/cgi-bin/own-disp?action=getissuer&CIK="+company_cik+r"&type=&dateb=&owner=include&start=0&count=1000"
    companyOwnerUrl = r"https://www.sec.gov/cgi-bin/own-disp?action=getowner&CIK="+company_cik

    # get company issuer html data
    issuerResponse = requests.get(url=companyIssuerUrl, allow_redirects=True, headers={"user-agent":"Rocio Jimenez jimenez.r.aa@m.titech.ac.jp"})
    # get company owner html data
    ownerResponse = requests.get(url=companyOwnerUrl, allow_redirects=True, headers={"user-agent":"Rocio Jimenez jimenez.r.aa@m.titech.ac.jp"})

    # BeautifulSoup
    ownerSoup = BeautifulSoup(ownerResponse.content, 'lxml')
    # print(ownerSoup)
    issuerSoup = BeautifulSoup(issuerResponse.content, 'lxml')
    # print(issuerSoup)

    tableCompany_owner = ownerSoup.find('table', attrs={"id":"transaction-report"})
    #print(tableCompany_owner)
    tableCompany_issuer = issuerSoup.find('table', attrs={"id":"transaction-report"})
    #print(tableCompany_issuer)

    if tableCompany_issuer:
        # issuer_table = pd.read_html(str(tablesCompany_issuer),header=0)[2]
        issuer_table = pd.read_html(str(tableCompany_issuer),header=0)[0]
    else:
        issuer_table = pd.DataFrame()

    if tableCompany_owner:
        owner_table = pd.read_html(str(tableCompany_owner),header=0)[0]
    else:
        owner_table = pd.DataFrame()

    if year:
        issuer_table = issuer_table[issuer_table['Transaction Date'].str.contains(year)]
        owner_table = owner_table[owner_table['Transaction Date'].str.contains(year)]
    
    issuer_table=issuer_table.dropna(axis=1, how="all")
    owner_table=owner_table.dropna(axis=1, how="all")

    return issuer_table, owner_table

In [42]:
def get_parent_subsidiary_subgraph(cik):
    """
    Queries the KG for subsidiary-parent structure and formats the result in LLM-readble format.
    """
    subgraph_triplets = []
    
    query = f"""MATCH (n:Company {{cik: "{cik}"}})-[r:IS_PARTIAL_OWNER_OF]-(m:Company)
            WITH n, r, m, startNode(r) AS origen
            RETURN 
                CASE WHEN origen.cik = n.cik THEN n.name ELSE m.name END AS Source_Name,
                CASE WHEN origen.cik = n.cik THEN m.name ELSE n.name END AS Target_Name,
                r AS Full_Rel"""

    with driver.session() as session:
        records = session.run(query, cik=cik)
    
        for record in records:
            name_init = record['Source_Name']
            name_last = record['Target_Name']
            
            relacion = record['Full_Rel']
            
            rel_tipo = relacion.type
            rel_props_str = json.dumps(dict(relacion)) 
            rel_repr = f"-[{rel_tipo} | Props: {rel_props_str}]->"
            
            tripleta = f"({name_init}) {rel_repr} ({name_last})"
            subgraph_triplets.append(tripleta)
            
    return subgraph_triplets

In [43]:
# for v3 report
def get_event_subgraph(cik, impact_rel="IMPACTS_STRICT_CORRECT"):

    def node_props(node):
        return dict(node) if node else None

    triples = set()  
    sector_written = False

    CYPHER_QUERY = f"""
    MATCH (company:Company {{cik: "{cik}"}})
    MATCH (company)-[r1:BELONGS_TO_INDUSTRY_OF]->(industry:StandardIndustrialClassification)

    MATCH (industry)<-[r2:BELONGS_TO_INDUSTRY_OF]-(peer:Company)
    MATCH (event2:Event)-[r4:{impact_rel}]->(peer)

    OPTIONAL MATCH (event1:Event)-[r3:{impact_rel}]->(company)
    OPTIONAL MATCH (event1)-[r5:EVENT_HAS_CATEGORY]->(etype1:EventCategory)
    OPTIONAL MATCH (event2)-[r6:EVENT_HAS_CATEGORY]->(etype2:EventCategory)

    RETURN DISTINCT
        company, industry, peer, event1, event2, etype1, etype2
    """

    with driver.session() as session:
        records = session.run(CYPHER_QUERY)

        main_company_events = {}
        peer_events = {}

        for record in records:
            company = record["company"]["name"]
            peer = record["peer"]["name"]
            industry = record["industry"]["name"]

            if not sector_written:
                triples.add(f"Common sector: {industry}")
                sector_written = True

            e1 = record["event1"]
            if e1:
                event_id = e1.element_id
                if event_id not in main_company_events:
                    main_company_events[event_id] = {
                        "props": node_props(e1),
                        "company": company,
                        "categories": set()
                    }
                if record["etype1"]:
                    main_company_events[event_id]["categories"].add(record["etype1"]["name"])

            e2 = record["event2"]
            if e2:
                event_id = e2.element_id
                if event_id not in peer_events:
                    peer_events[event_id] = {
                        "props": node_props(e2),
                        "company": peer,
                        "categories": set()
                    }
                if record["etype2"]:
                    peer_events[event_id]["categories"].add(record["etype2"]["name"])

        for e in main_company_events.values():
            props = ', '.join(f"{k}: {v}" for k,v in e["props"].items())
            cats = sorted(list(e["categories"]))
            triples.add(f"{props} - impacts -> {e['company']} - categories: {cats}")

        for e in peer_events.values():
            props = ', '.join(f"{k}: {v}" for k,v in e["props"].items())
            cats = sorted(list(e["categories"]))
            triples.add(f"{props} - impacts -> {e['company']} - categories: {cats}")

    return "\n".join(sorted(triples))

print(get_event_subgraph("0001744489", impact_rel="IMPACTS_CORRECT"))

Common sector: SERVICES-MISCELLANEOUS AMUSEMENT & RECREATION
date: None, geo: ['United States'], description: Nineteen Al-Qaeda terrorists hijack four planes, crashing two into the twin towers of the World Trade Center in New York City, the third plane into the Pentagon in Washington, DC, while the fourth plane is downed on the outskirts of Stonycreek Township, Pennsylvania. 2,996 people, including 2,977 victims and 19 hijackers, die in the attacks., id: 2001_6 - impacts -> Walt Disney Co - categories: ['terrorist_attacks']
date: ['2001-10-23 00:00:00', '2001-10-23 23:59:59'], geo: [], description: Steve Jobs introduces the first iPod., id: 2001_8 - impacts -> Walt Disney Co - categories: ['consumer_tech_milestones']
date: ['2002-03-14 00:00:00', '2002-03-14 23:59:59'], geo: [], description: SpaceX is founded by Elon Musk., id: 2002_3 - impacts -> Walt Disney Co - categories: ['disruptive_technologies', 'founding_of_major_companies']
date: ['2002-11-16 00:00:00', '2002-11-16 23:59:59']

## Dataset Generation

In [44]:
df = pd.DataFrame(columns=["company","year","v0","v1","v2","v3","v_all"])
df

,company,year,v0,v1,v2,v3,v_all


### query for viewing the news as well, use this in the neo4j browser

In [45]:
"""
MATCH (company:Company {cik: "0001070985"})
MATCH (company)-[r1:BELONGS_TO_INDUSTRY_OF]->(industry:StandardIndustrialClassification)

MATCH (industry)<-[r2:BELONGS_TO_INDUSTRY_OF]-(peer:Company)
MATCH (event2:Event)-[r4:IMPACTS_STRICT_CORRECT]->(peer)

OPTIONAL MATCH (event1:Event)-[r3:IMPACTS_STRICT_CORRECT]->(company)

OPTIONAL MATCH (event1)-[r5:EVENT_HAS_CATEGORY]->(etype1:EventCategory)
OPTIONAL MATCH (event2)-[r6:EVENT_HAS_CATEGORY]->(etype2:EventCategory)

OPTIONAL MATCH (company)<-[r7:IS_PARTIAL_OWNER_OF]-(owner:Company)
OPTIONAL MATCH (company)-[r8:IS_PARTIAL_OWNER_OF]->(owned:Company)

OPTIONAL MATCH (event3:Event)-[r9:IMPACTS_STRICT_CORRECT]->(owner)
OPTIONAL MATCH (event4:Event)-[r10:IMPACTS_STRICT_CORRECT]->(owned)

OPTIONAL MATCH (event3)-[r11:EVENT_HAS_CATEGORY]->(etype3:EventCategory)
OPTIONAL MATCH (event4)-[r12:EVENT_HAS_CATEGORY]->(etype4:EventCategory)

MATCH (company)-[r13:HAS_STATE_LOCATION]->(state:State)
MATCH (state)<-[r14:HAS_STATE_LOCATION]-(neighbor:Company)
MATCH (event5:Event)-[r15:IMPACTS_STRICT_CORRECT]->(neighbor)
OPTIONAL MATCH (event5)-[r16:EVENT_HAS_CATEGORY]->(etype5:EventCategory)

//
// Supercategorías conectadas a las categorías originales
//
OPTIONAL MATCH (etype1)-[rSuper1:SUBCATEGORY_OF*0..]->(supertype1:EventCategory)
OPTIONAL MATCH (etype2)-[rSuper2:SUBCATEGORY_OF*0..]->(supertype2:EventCategory)
OPTIONAL MATCH (etype3)-[rSuper3:SUBCATEGORY_OF*0..]->(supertype3:EventCategory)
OPTIONAL MATCH (etype4)-[rSuper4:SUBCATEGORY_OF*0..]->(supertype4:EventCategory)
OPTIONAL MATCH (etype5)-[rSuper5:SUBCATEGORY_OF*0..]->(supertype5:EventCategory)

//
// Noticias que mencionan tanto el evento como la compañía afectada
//
OPTIONAL MATCH (news1:News)-[rNewsEvent1:MENTIONS]->(event1),
               (news1)-[rNewsComp1:MENTIONS]->(company)
OPTIONAL MATCH (news2:News)-[rNewsEvent2:MENTIONS]->(event2),
               (news2)-[rNewsComp2:MENTIONS]->(peer)
OPTIONAL MATCH (news3:News)-[rNewsEvent3:MENTIONS]->(event3),
               (news3)-[rNewsComp3:MENTIONS]->(owner)
OPTIONAL MATCH (news4:News)-[rNewsEvent4:MENTIONS]->(event4),
               (news4)-[rNewsComp4:MENTIONS]->(owned)
OPTIONAL MATCH (news5:News)-[rNewsEvent5:MENTIONS]->(event5),
               (news5)-[rNewsComp5:MENTIONS]->(neighbor)

WITH company, industry, peer, state,
     owner, owned, neighbor,
     event1, event2, event3, event4, event5,
     etype1, etype2, etype3, etype4, etype5,
     supertype1, supertype2, supertype3, supertype4, supertype5,
     news1, news2, news3, news4, news5,
     r1, r2, r3, r4, r5, r6, r7, r8, r9, r10, r11, r12,
     r13, r14, r15, r16,
     rSuper1, rSuper2, rSuper3, rSuper4, rSuper5,
     rNewsEvent1, rNewsComp1,
     rNewsEvent2, rNewsComp2,
     rNewsEvent3, rNewsComp3,
     rNewsEvent4, rNewsComp4,
     rNewsEvent5, rNewsComp5
WHERE event1 IS NOT NULL 
   OR event2 IS NOT NULL 
   OR event3 IS NOT NULL 
   OR event4 IS NOT NULL 
   OR event5 IS NOT NULL

RETURN DISTINCT
  company, industry, state,
  peer, owner, owned, neighbor,
  event1, event2, event3, event4, event5,
  etype1, etype2, etype3, etype4, etype5,
  supertype1, supertype2, supertype3, supertype4, supertype5,
  news1, news2, news3, news4, news5,
  r1, r2, r3, r4, r5, r6, r7, r8, r9, r10, r11, r12,
  r13, r14, r15, r16,
  rSuper1, rSuper2, rSuper3, rSuper4, rSuper5,
  rNewsEvent1, rNewsComp1,
  rNewsEvent2, rNewsComp2,
  rNewsEvent3, rNewsComp3,
  rNewsEvent4, rNewsComp4,
  rNewsEvent5, rNewsComp5"""

'\nMATCH (company:Company {cik: "0001070985"})\nMATCH (company)-[r1:BELONGS_TO_INDUSTRY_OF]->(industry:StandardIndustrialClassification)\n\nMATCH (industry)<-[r2:BELONGS_TO_INDUSTRY_OF]-(peer:Company)\nMATCH (event2:Event)-[r4:IMPACTS_STRICT_CORRECT]->(peer)\n\nOPTIONAL MATCH (event1:Event)-[r3:IMPACTS_STRICT_CORRECT]->(company)\n\nOPTIONAL MATCH (event1)-[r5:EVENT_HAS_CATEGORY]->(etype1:EventCategory)\nOPTIONAL MATCH (event2)-[r6:EVENT_HAS_CATEGORY]->(etype2:EventCategory)\n\nOPTIONAL MATCH (company)<-[r7:IS_PARTIAL_OWNER_OF]-(owner:Company)\nOPTIONAL MATCH (company)-[r8:IS_PARTIAL_OWNER_OF]->(owned:Company)\n\nOPTIONAL MATCH (event3:Event)-[r9:IMPACTS_STRICT_CORRECT]->(owner)\nOPTIONAL MATCH (event4:Event)-[r10:IMPACTS_STRICT_CORRECT]->(owned)\n\nOPTIONAL MATCH (event3)-[r11:EVENT_HAS_CATEGORY]->(etype3:EventCategory)\nOPTIONAL MATCH (event4)-[r12:EVENT_HAS_CATEGORY]->(etype4:EventCategory)\n\nMATCH (company)-[r13:HAS_STATE_LOCATION]->(state:State)\nMATCH (state)<-[r14:HAS_STATE_LOCA

### Defining data, questions and other text variables

In [ ]:
## dummy data
company = "PayPal"
company_ticker = "PYPL"
company_cik = "0001633917"
year = "2024"


sample_report = """### Key Rating Drivers (Example)

- Strong Recent Performance: During the past two years, EBITDA increased to <$Value> in <$Year1> and <$Value> in <$Year2> compared to around <$Value> in <$Year0>. This increase was supported by strong operating momentum with good demand trends for food, fuel and feed. Given the tight commodity supply environment, this resulted in good profit generation and structurally higher margins for global agribusiness companies, including <Company>.
- The deteriorating economic conditions in <Country> remain another key wildcard given the country's leading position as the world’s largest <Commodity> exporter and low-cost producer. It is anticipated that <Commodity> margins outside <Country> will benefit if exports remain lower than normal. <Country> is experiencing severe economic issues that caused the new government to implement various currency and capital controls, with increased export taxes on several commodities, including <Commodity>, as part of its plan to address solvency concerns. The increased taxes combined with stockpiling by farmers and the financial distress of one of the largest processors in <Country>, if unchanged during <$Year>, will materially reduce exports, with <Country A> and <Country B> expected to benefit. <Company> does not have any exposure to operations in <Country>.
- Growing Nutrition Business: <Company> increased its exposure to higher growth, value-added assets through M&A and/or organic investments, particularly targeting the nutrition segment. This includes acquisitions of <Company A> (<Segment A>), <Company B> (<Segment B>) and <Company C> (<Segment C>). Over the longer term, <Company> expects to increase the contribution from the nutrition segment, currently in the low double digits, to <$TargetPercentage>% of overall earnings through the growth and margin expansion of existing investments and bolt-on M&A, including increased exposure to health and wellness assets."""


## we select at least one company from each sector
# we also include one recent (2025 or 2024) and one older report
report_data_list = [
    {
        "company_name":"CoreCivic, Inc.",
        "company_cik":"0001070985",
        "company_ticker":"CXW",
        "year":"2022"
    },
    {
        "company_name":"CoreCivic, Inc.",
        "company_cik":"0001070985",
        "company_ticker":"CXW",
        "year":"2025"
    },
    {
        "company_name":"Walt Disney Co",
        "company_cik":"0001744489",
        "company_ticker":"DIS",
        "year":"2023"
    },
    {
        "company_name":"Walt Disney Co",
        "company_cik":"0001744489",
        "company_ticker":"DIS",
        "year":"2025"
    },
    {
        "company_name":"ALASKA AIR GROUP, INC.",
        "company_cik":"0000766421",
        "company_ticker":"ALK",
        "year":"2022"
    },
    {
        "company_name":"ALASKA AIR GROUP, INC.",
        "company_cik":"0000766421",
        "company_ticker":"ALK",
        "year":"2025"
    },
    {
        "company_name":"Merck & Co., Inc.",
        "company_cik":"0000310158",
        "company_ticker":"MRK",
        "year":"2022"
    },
    {
        "company_name":"Merck & Co., Inc.",
        "company_cik":"0000310158",
        "company_ticker":"MRK",
        "year":"2025"
    },
    {
        "company_name":"OCCIDENTAL PETROLEUM CORP",
        "company_cik":"0000797468",
        "company_ticker":"OXY",
        "year":"2023"
    },
    {
        "company_name":"OCCIDENTAL PETROLEUM CORP",
        "company_cik":"0000797468",
        "company_ticker":"OXY",
        "year":"2025"
    },
]

# define the question lists for each type of factor
f1_questions = """
**Profitability / Margin**
1. How has EBITDA margin trended over the past years, and does it indicate sufficient profitability to support debt repayment?
2. How does the company's profitability compare with peers within the same sector?
**Cash Flow**
3. Are FFO, CFO, and FCF consistent, and do they demonstrate sufficient cash generation to cover operational and financial obligations?
4. Are there signs of deterioration in cash conversion efficiency (EBITDA → CFO → FCF)?
**Coverage**
5. Is the interest  coverage ratio adequate to withstand potential declines in earnings?
6. How would coverage ratios change under a 10-20% decrease in FFO or CFO?
**Leverage**
7. What is the level and trend of leverage (Debt/FFO or Debt/EBITDA), and is it sustainable relative to cash generation?
8. How does near-term refinancing risk appear when evaluating Cash/ST Debt and Free Cash Flow ratio?
**Liquidity**
9. Does the company have sufficient cash and short-term resources to cover immediate obligations?
10. Are there potential liquidity pressures in the near term under declining cash flow scenarios (using Cash/ST Debt and FCF)?
**Efficiency / Cash Quality**
11. Are there discrepancies between EBITDA, CFO, and FCF that could indicate issues in cash generation quality?
12. Are there divergences between EBITDA growth and FCF growth, and what might they reveal about CapEx or working capital management?
**Sector / Peer Analysis**
13. How do the company's key metrics (EBITDA margin, FFO, leverage, liquidity) compare with peers within the same sector?
**Scenario / Stress Test**
14. Under adverse scenarios (e.g., reduced cash flow), do coverage, leverage, and liquidity metrics remain adequate?
***Future Predictions***
15 Based on the available data, provide a general forecast on the company's stability and credit risk."""

f2_questions = """
- *Do transaction patterns (volume, frequency, type) reveal a strategic shift or a reaction to specific events? Do significant spikes in activity correlate with major corporate announcements, suggesting insiders may have non-public information?*
- *Is activity concentrated among key executives (e.g., CEO, CFO) or significant shareholders? Does a decline in insider stakes signal a lack of long-term confidence or simply portfolio diversification?*
- *Is ownership becoming more concentrated (increasing influence) or dispersed (losing control)? Do these changes align with management or board reshuffles, suggesting a shift in strategic vision or alignment of interests?*
- *Is the issuance of new stock options or awards excessive, creating dilution risk for existing shareholders? Does a drop in insider ownership correlate with compensation packages that might be misaligned with shareholder interests?*

Only if there is information about the Group Structure:
- Consider whether the group structure has any impact on the credit risk. For instance, a weak subsidiary can be potencially supported by a strong parent. Inversely, a stronger subsidiary could be vulnerable to actions from a weaker parent to divert some of its resources.
"""

f3_questions = f"""
- Recent event impact: What events from this year {year} and the previous 3 years have impacted the company recently?
- Full event impact history: Which events have impacted the company in its history and which of these events had the greatest impact? Which categories do these events belong to?
- What other events have affected companies that belong to the same sector as {company}? What are their event categories? Is the company exposed to risks related to these events as well or is it resilient enough to deal and adapt to them?
    
With this information, think about, what events are likely to positively or negatively impact the companies credit risk and which events will not impact it.
"""

# queries for generating subgraph
# WE ARE NOT USING THESE

# simple query, only direct events impacting the company and its peers
query = """
MATCH (company:Company {cik: $cik})
MATCH (company)-[r1:BELONGS_TO_INDUSTRY_OF]->(industry:StandardIndustrialClassification)

MATCH (industry)<-[r2:BELONGS_TO_INDUSTRY_OF]-(peer:Company)
MATCH (event2:Event)-[r4:IMPACTS_STRICT_CORRECT]->(peer)

OPTIONAL MATCH (event1:Event)-[r3:IMPACTS_STRICT_CORRECT]->(company)

OPTIONAL MATCH (event1)-[r5:EVENT_HAS_CATEGORY]->(etype1:EventCategory)
OPTIONAL MATCH (event2)-[r6:EVENT_HAS_CATEGORY]->(etype2:EventCategory)

RETURN DISTINCT
  company, industry, peer, event1, event2, etype1, etype2,
  r1, r2, r3, r4, r5, r6
"""

# include event supercategories and events impacting owners and owned companies as well
query_complex = """
MATCH (company:Company {cik: $cik})
MATCH (company)-[r1:BELONGS_TO_INDUSTRY_OF]->(industry:StandardIndustrialClassification)

MATCH (industry)<-[r2:BELONGS_TO_INDUSTRY_OF]-(peer:Company)
MATCH (event2:Event)-[r4:IMPACTS_CORRECT]->(peer)

OPTIONAL MATCH (event1:Event)-[r3:IMPACTS_CORRECT]->(company)

OPTIONAL MATCH (event1)-[r5:EVENT_HAS_CATEGORY]->(etype1:EventCategory)
OPTIONAL MATCH (event2)-[r6:EVENT_HAS_CATEGORY]->(etype2:EventCategory)

OPTIONAL MATCH (company)<-[r7:IS_PARTIAL_OWNER_OF]-(owner:Company)
OPTIONAL MATCH (company)-[r8:IS_PARTIAL_OWNER_OF]->(owned:Company)

OPTIONAL MATCH (event3:Event)-[r9:IMPACTS_CORRECT]->(owner)
OPTIONAL MATCH (event4:Event)-[r10:IMPACTS_CORRECT]->(owned)

OPTIONAL MATCH (event3)-[r11:EVENT_HAS_CATEGORY]->(etype3:EventCategory)
OPTIONAL MATCH (event4)-[r12:EVENT_HAS_CATEGORY]->(etype4:EventCategory)

OPTIONAL MATCH (etype1)-[rSuper1:SUBCATEGORY_OF*0..]->(supertype1:EventCategory)
OPTIONAL MATCH (etype2)-[rSuper2:SUBCATEGORY_OF*0..]->(supertype2:EventCategory)
OPTIONAL MATCH (etype3)-[rSuper3:SUBCATEGORY_OF*0..]->(supertype3:EventCategory)
OPTIONAL MATCH (etype4)-[rSuper4:SUBCATEGORY_OF*0..]->(supertype4:EventCategory)

WITH company, industry, peer, event1, event2, etype1, etype2,
     owner, owned, event3, event4, etype3, etype4,
     r1, r2, r3, r4, r5, r6, r7, r8, r9, r10, r11, r12,
     supertype1, supertype2, supertype3, supertype4,
     rSuper1, rSuper2, rSuper3, rSuper4
WHERE event3 IS NOT NULL OR event4 IS NOT NULL OR event1 IS NOT NULL OR event2 IS NOT NULL

RETURN DISTINCT
  company, industry, peer, owner, owned,
  event1, event2, event3, event4,
  etype1, etype2, etype3, etype4,
  supertype1, supertype2, supertype3, supertype4,
  r1, r2, r3, r4, r5, r6, r7, r8, r9, r10, r11, r12,
  rSuper1, rSuper2, rSuper3, rSuper4
  """

# same but using the strict threshold event set
query_complex_strict = """
MATCH (company:Company {cik: $cik})
MATCH (company)-[r1:BELONGS_TO_INDUSTRY_OF]->(industry:StandardIndustrialClassification)

MATCH (industry)<-[r2:BELONGS_TO_INDUSTRY_OF]-(peer:Company)
MATCH (event2:Event)-[r4:IMPACTS_STRICT_CORRECT]->(peer)

OPTIONAL MATCH (event1:Event)-[r3:IMPACTS_STRICT_CORRECT]->(company)

OPTIONAL MATCH (event1)-[r5:EVENT_HAS_CATEGORY]->(etype1:EventCategory)
OPTIONAL MATCH (event2)-[r6:EVENT_HAS_CATEGORY]->(etype2:EventCategory)

OPTIONAL MATCH (company)<-[r7:IS_PARTIAL_OWNER_OF]-(owner:Company)
OPTIONAL MATCH (company)-[r8:IS_PARTIAL_OWNER_OF]->(owned:Company)

OPTIONAL MATCH (event3:Event)-[r9:IMPACTS_STRICT_CORRECT]->(owner)
OPTIONAL MATCH (event4:Event)-[r10:IMPACTS_STRICT_CORRECT]->(owned)

OPTIONAL MATCH (event3)-[r11:EVENT_HAS_CATEGORY]->(etype3:EventCategory)
OPTIONAL MATCH (event4)-[r12:EVENT_HAS_CATEGORY]->(etype4:EventCategory)

OPTIONAL MATCH (etype1)-[rSuper1:SUBCATEGORY_OF*0..]->(supertype1:EventCategory)
OPTIONAL MATCH (etype2)-[rSuper2:SUBCATEGORY_OF*0..]->(supertype2:EventCategory)
OPTIONAL MATCH (etype3)-[rSuper3:SUBCATEGORY_OF*0..]->(supertype3:EventCategory)
OPTIONAL MATCH (etype4)-[rSuper4:SUBCATEGORY_OF*0..]->(supertype4:EventCategory)

WITH company, industry, peer, event1, event2, etype1, etype2,
     owner, owned, event3, event4, etype3, etype4,
     r1, r2, r3, r4, r5, r6, r7, r8, r9, r10, r11, r12,
     supertype1, supertype2, supertype3, supertype4,
     rSuper1, rSuper2, rSuper3, rSuper4
WHERE event3 IS NOT NULL OR event4 IS NOT NULL OR event1 IS NOT NULL OR event2 IS NOT NULL

RETURN DISTINCT
  company, industry, peer, owner, owned,
  event1, event2, event3, event4,
  etype1, etype2, etype3, etype4,
  supertype1, supertype2, supertype3, supertype4,
  r1, r2, r3, r4, r5, r6, r7, r8, r9, r10, r11, r12,
  rSuper1, rSuper2, rSuper3, rSuper4
  """

# adds events impacting neighboring companies in the same state location
query_complex_states = """
MATCH (company:Company {cik: $cik})
MATCH (company)-[r1:BELONGS_TO_INDUSTRY_OF]->(industry:StandardIndustrialClassification)

MATCH (industry)<-[r2:BELONGS_TO_INDUSTRY_OF]-(peer:Company)
MATCH (event2:Event)-[r4:IMPACTS_STRICT_CORRECT]->(peer)

OPTIONAL MATCH (event1:Event)-[r3:IMPACTS_STRICT_CORRECT]->(company)

OPTIONAL MATCH (event1)-[r5:EVENT_HAS_CATEGORY]->(etype1:EventCategory)
OPTIONAL MATCH (event2)-[r6:EVENT_HAS_CATEGORY]->(etype2:EventCategory)

OPTIONAL MATCH (company)<-[r7:IS_PARTIAL_OWNER_OF]-(owner:Company)
OPTIONAL MATCH (company)-[r8:IS_PARTIAL_OWNER_OF]->(owned:Company)

OPTIONAL MATCH (event3:Event)-[r9:IMPACTS_STRICT_CORRECT]->(owner)
OPTIONAL MATCH (event4:Event)-[r10:IMPACTS_STRICT_CORRECT]->(owned)

OPTIONAL MATCH (event3)-[r11:EVENT_HAS_CATEGORY]->(etype3:EventCategory)
OPTIONAL MATCH (event4)-[r12:EVENT_HAS_CATEGORY]->(etype4:EventCategory)

MATCH (company)-[r13:HAS_STATE_LOCATION]->(state:State)
MATCH (state)<-[r14:HAS_STATE_LOCATION]-(neighbor:Company)
MATCH (event5:Event)-[r15:IMPACTS_STRICT_CORRECT]->(neighbor)
OPTIONAL MATCH (event5)-[r16:EVENT_HAS_CATEGORY]->(etype5:EventCategory)

OPTIONAL MATCH (etype1)-[rSuper1:SUBCATEGORY_OF*0..]->(supertype1:EventCategory)
OPTIONAL MATCH (etype2)-[rSuper2:SUBCATEGORY_OF*0..]->(supertype2:EventCategory)
OPTIONAL MATCH (etype3)-[rSuper3:SUBCATEGORY_OF*0..]->(supertype3:EventCategory)
OPTIONAL MATCH (etype4)-[rSuper4:SUBCATEGORY_OF*0..]->(supertype4:EventCategory)
OPTIONAL MATCH (etype5)-[rSuper5:SUBCATEGORY_OF*0..]->(supertype5:EventCategory)

WITH company, industry, peer, state,
     owner, owned, neighbor,
     event1, event2, event3, event4, event5,
     etype1, etype2, etype3, etype4, etype5,
     r1, r2, r3, r4, r5, r6, r7, r8, r9, r10, r11, r12,
     r13, r14, r15, r16,
     supertype1, supertype2, supertype3, supertype4, supertype5,
     rSuper1, rSuper2, rSuper3, rSuper4, rSuper5

WHERE event1 IS NOT NULL 
   OR event2 IS NOT NULL 
   OR event3 IS NOT NULL 
   OR event4 IS NOT NULL 
   OR event5 IS NOT NULL

RETURN DISTINCT
  company, industry, state,
  peer, owner, owned, neighbor,
  event1, event2, event3, event4, event5,
  etype1, etype2, etype3, etype4, etype5,
  supertype1, supertype2, supertype3, supertype4, supertype5,
  r1, r2, r3, r4, r5, r6, r7, r8, r9, r10, r11, r12,
  r13, r14, r15, r16,
  rSuper1, rSuper2, rSuper3, rSuper4, rSuper5
  """

### main code

In [ ]:
## MAIN CODE
for report in report_data_list:
    
    # print("report for",report["company_name"],report["year"])
    
    # generate v0 report
    report_v0 = generate_report(company=report["company_name"],
                                company_cik=report["company_cik"],
                                year=report["year"],
                                prompt_type="v0",
                                self_correct=False)
    
    print("# V0")
    print("---------------------------------------------------")
    print(report_v0)
    print("---------------------------------------------------")
    # print("V0 corrected")
    # print("---------------------------------------------------")
    # print(report_v0_corrected)    

    # generate v1 report
    # get the quantitative data
    quantitative_data = get_fitch_metrics_timeseries(report["company_ticker"], report_year=int(report["year"]))

    # get peer data for peer comparison
    peer_name, peer_ticker = get_peer_data(report["company_cik"],report["company_ticker"])
    peer_quantitative_data = get_fitch_metrics_timeseries(peer_ticker, report_year=int(report["year"]))

    report_v1, report_v1_corrected = generate_report(company=report["company_name"],
                                company_cik=report["company_cik"],
                                year=report["year"],
                                prompt_type="v1",
                                questions=f1_questions,
                                quantitative_data = quantitative_data.to_markdown(),
                                peer_quantitative_data = peer_quantitative_data.to_markdown(),
                                peer_company_name=peer_name)

    print("# V1")
    print("---------------------------------------------------")
    print(quantitative_data.to_markdown())
    print("---------------------------------------------------")
    print(report_v1)
    print("# V1 corrected")
    print("---------------------------------------------------")
    print(report_v1_corrected)

    # generate v2 report
    # get the parent-subsidiary structure
    parent_subsidiary_triples = get_parent_subsidiary_subgraph(report["company_cik"])

    # get the issuer and owner tables
    issuer_table, owner_table = get_issuer_owner_tables(report["company_cik"],year=report["year"])

    report_v2,report_v2_corrected = generate_report(company=report["company_name"],
                                company_cik=report["company_cik"], 
                                year=report["year"],
                                prompt_type="v2",
                                issuer_table=issuer_table,
                                owner_table=owner_table,
                                parent_subsidiary_triples=parent_subsidiary_triples,
                                questions=f2_questions)
    
    print("# V2")
    print("---------------------------------------------------")
    print("#### ISSUEAR TABLE")
    print(issuer_table.to_markdown())
    print("---------------------------------------------------")
    print("#### OWNER TABLE")
    print(owner_table.to_markdown())
    print("---------------------------------------------------")
    print(report_v2)
    print("# V2 corrected")
    print("---------------------------------------------------")
    print(report_v2_corrected)

    # generate v3 report
    # get the subgraph
    subgraph = get_event_subgraph(report["company_cik"], impact_rel="IMPACTS_CORRECT") # using the less strict confidence set

    report_v3, report_v3_corrected = generate_report(company=report["company_name"],
                                company_cik=report["company_cik"], 
                                year=report["year"],
                                prompt_type="v3",
                                subgraph_triples=subgraph,
                                questions=f3_questions)
    
    print("# V3")
    print("---------------------------------------------------")
    print(report_v3)
    print("# V3 corrected")
    print("---------------------------------------------------")
    print(report_v3_corrected)

    # generate final report
    final_report = generate_report(company=report["company_name"],
                                company_cik=report["company_cik"], 
                                year=report["year"],
                                prompt_type="v_all",
                                report_v1=report_v1_corrected,
                                report_v2=report_v2_corrected,
                                report_v3=report_v3_corrected,
                                self_correct=False)

    print("# FINAL")
    print("---------------------------------------------------")
    print(final_report)
    # print("FINAL corrected")
    # print("---------------------------------------------------")
    # print(final_report_corrected)

    # append to dataframe
    df = pd.concat([df, pd.DataFrame([{
        "company": report["company_name"],
        "year": report["year"],
        "v0": report_v0,
        "v0 corrected": "",
        "v1": report_v1,
        "v1 corrected": report_v1_corrected,
        "v2": report_v2,
        "v2 corrected": report_v2_corrected,
        "v3": report_v3,
        "v3 corrected": report_v3_corrected,
        "v_all": final_report,
        "v_all corrected": ""
        }])], ignore_index=True)


# V0
---------------------------------------------------
**Key Rating Drivers for CoreCivic, Inc. – 2022**

---

**F1: Elevated Leverage Constrains Financial Flexibility**  
CoreCivic’s credit profile in 2022 is significantly weighed down by its relatively high leverage levels. The company's debt-to-EBITDA ratio remains elevated compared with industry norms, limiting CoreCivic’s ability to absorb shocks or invest in growth opportunities without increasing risk. This capital structure tightness restricts operational flexibility and poses refinancing risks in an interest rate environment characterized by upward pressure. Furthermore, persistent margin pressures have hindered EBITDA growth, exacerbating leverage concerns and increasing sensitivity to cash flow volatility.

---

**F1: Volatile Profitability Amid Contract Uncertainty**  
Profitability metrics declined notably in 2022 due to contract term renewals and government budget pressures, directly impacting revenue visibility. CoreCi

/tmp/ipykernel_25200/3378488874.py:23: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  issuer_table = pd.read_html(str(tableCompany_issuer),header=0)[0]
/tmp/ipykernel_25200/3378488874.py:28: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  owner_table = pd.read_html(str(tableCompany_owner),header=0)[0]


# V2
---------------------------------------------------
#### ISSUEAR TABLE
|     | Acquistion or Disposition   | Transaction Date   |   Deemed Execution Date | Reporting Owner       |   Form | Transaction Type   | Direct or Indirect Ownership   |   Number of Securities Transacted |   Number of Securities Owned |   Line Number |   Owner CIK | Security Name   |
|----:|:----------------------------|:-------------------|------------------------:|:----------------------|-------:|:-------------------|:-------------------------------|----------------------------------:|-----------------------------:|--------------:|------------:|:----------------|
| 172 | D                           | 2022-12-16         |                     nan | Lappin Harley G.      |      4 | S-Sale             | --D                            |                              2000 |                        71475 |             1 |         nan | Common Stock    |
| 173 | D                           | 2022-12-15         |     

/tmp/ipykernel_25200/3378488874.py:23: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  issuer_table = pd.read_html(str(tableCompany_issuer),header=0)[0]
/tmp/ipykernel_25200/3378488874.py:28: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  owner_table = pd.read_html(str(tableCompany_owner),header=0)[0]


# V2
---------------------------------------------------
#### ISSUEAR TABLE
|    | Acquistion or Disposition   | Transaction Date   |   Deemed Execution Date | Reporting Owner            |   Form | Transaction Type   | Direct or Indirect Ownership   |   Number of Securities Transacted |   Number of Securities Owned |   Line Number |   Owner CIK | Security Name   |
|---:|:----------------------------|:-------------------|------------------------:|:---------------------------|-------:|:-------------------|:-------------------------------|----------------------------------:|-----------------------------:|--------------:|------------:|:----------------|
|  0 | D                           | 2025-09-11         |                     nan | Grande Anthony L           |      4 | S-Sale             | --D                            |                             22500 |             135559           |             1 |         nan | Common Stock    |
|  1 | D                           | 2025-09-09    

...

# V2
---------------------------------------------------
#### ISSUEAR TABLE
|    | Acquistion or Disposition   | Transaction Date   |   Deemed Execution Date | Reporting Owner              |   Form | Transaction Type   | Direct or Indirect Ownership   |   Number of Securities Transacted |   Number of Securities Owned |   Line Number |   Owner CIK | Security Name               |
|---:|:----------------------------|:-------------------|------------------------:|:-----------------------------|-------:|:-------------------|:-------------------------------|----------------------------------:|-----------------------------:|--------------:|------------:|:----------------------------|
|  0 | A                           | 2025-09-30         |                     nan | Coe Mary Ellen               |      4 | A-Award            | --D                            |                          357.441  |                    28684.7   |             2 |         nan | Phantom Stock               |
|  1 | A 

/tmp/ipykernel_25200/3378488874.py:23: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  issuer_table = pd.read_html(str(tableCompany_issuer),header=0)[0]
/tmp/ipykernel_25200/3378488874.py:28: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  owner_table = pd.read_html(str(tableCompany_owner),header=0)[0]


# V2
---------------------------------------------------
#### ISSUEAR TABLE
|     | Acquistion or Disposition   | Transaction Date   |   Deemed Execution Date | Reporting Owner         |   Form | Transaction Type   | Direct or Indirect Ownership   |   Number of Securities Transacted |   Number of Securities Owned |   Line Number |   Owner CIK | Security Name           |
|----:|:----------------------------|:-------------------|------------------------:|:------------------------|-------:|:-------------------|:-------------------------------|----------------------------------:|-----------------------------:|--------------:|------------:|:------------------------|
| 168 | A                           | 2023-12-21         |                     nan | BERKSHIRE HATHAWAY INC  |      4 | P-Purchase         | --I                            |                       1.74312e+06 |                  2.43716e+08 |             4 |         nan | Common Stock            |
| 169 | A                        

/tmp/ipykernel_25200/3378488874.py:23: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  issuer_table = pd.read_html(str(tableCompany_issuer),header=0)[0]
/tmp/ipykernel_25200/3378488874.py:28: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  owner_table = pd.read_html(str(tableCompany_owner),header=0)[0]


# V2
---------------------------------------------------
#### ISSUEAR TABLE
|    | Acquistion or Disposition   | Transaction Date   |   Deemed Execution Date | Reporting Owner         |   Form | Transaction Type   | Direct or Indirect Ownership   |   Number of Securities Transacted |   Number of Securities Owned |   Line Number |   Owner CIK | Security Name           |
|---:|:----------------------------|:-------------------|------------------------:|:------------------------|-------:|:-------------------|:-------------------------------|----------------------------------:|-----------------------------:|--------------:|------------:|:------------------------|
|  0 | D                           | 2025-10-02         |                     nan | Kerrigan Sylvia J       |      4 | F-InKind           | --D                            |                              5324 |             148837           |             1 |         nan | Common Stock            |
|  1 | A                           |

In [ ]:
df

,company,year,v0,v1,v2,v3,v_all,v0 corrected,v1 corrected,v2 corrected,v3 corrected,v_all corrected
0,"CoreCivic, Inc.",2022,### Key Rating Drivers\n\n- **Stable Governmen...,### Key Rating Drivers\n\n- **Moderate EBITDA ...,### Key Rating Drivers\n\n- Moderate Insider T...,### Key Rating Drivers\n\n- Stable Government ...,### Key Rating Drivers\n\n- **Volatile but Rec...,### Key Rating Drivers\n\n- **Revenue Stabilit...,### Key Rating Drivers\n\n- **Declining EBITDA...,### Key Rating Drivers\n\n- Insider Transactio...,### Key Rating Drivers\n\n- **Stable Governmen...,### Key Rating Drivers\n\n- **Declining EBITDA...
1,"CoreCivic, Inc.",2025,### Key Rating Drivers\n\n- Stable Government ...,"### Key Rating Drivers – CoreCivic, Inc. (2025...",### Key Rating Drivers\n\n- Elevated Insider S...,### Key Rating Drivers\n\n- Stable Government-...,"### Key Rating Drivers – CoreCivic, Inc. (2025...",### Key Rating Drivers\n\n- **Stable Contractu...,"### Key Rating Drivers – CoreCivic, Inc. (2025...",### Key Rating Drivers\n\n- Elevated Insider S...,### Key Rating Drivers\n\n- Contractual Revenu...,"### Key Rating Drivers – CoreCivic, Inc. (2025..."
2,Walt Disney Co,2023,### Key Rating Drivers – Walt Disney Co (CIK 0...,### Key Rating Drivers – Walt Disney Co (2023)...,"### Key Rating Drivers – Walt Disney Co, 2023\...",### Key Rating Drivers – Walt Disney Co (2023)...,### Key Rating Drivers – Walt Disney Co (2023)...,### Key Rating Drivers – Walt Disney Co (CIK 0...,### Key Rating Drivers – Walt Disney Co (2023)...,"### Key Rating Drivers – Walt Disney Co, 2023\...",### Key Rating Drivers – Walt Disney Co (2023)...,### Key Rating Drivers – Walt Disney Co (2023)...
3,Walt Disney Co,2025,### Key Rating Drivers – Walt Disney Co (CIK 0...,### Key Rating Drivers\n\n- **Sustained EBITDA...,### Key Rating Drivers\n\n- Moderate Insider T...,### Key Rating Drivers – Walt Disney Co (2025)...,### Key Rating Drivers – Walt Disney Co (2025)...,### Key Rating Drivers – Walt Disney Co (CIK 0...,### Key Rating Drivers\n\n- **Improving EBITDA...,### Key Rating Drivers\n\n- Consistent Insider...,### Key Rating Drivers – Walt Disney Co (2025)...,### Key Rating Drivers – Walt Disney Co (2025)...
4,"ALASKA AIR GROUP, INC.",2022,### Key Rating Drivers\n\n- **Solid Market Pos...,"### Key Rating Drivers – ALASKA AIR GROUP, INC...",### Key Rating Drivers\n\n- Moderate Insider T...,### Key Rating Drivers\n\n- Moderate Exposure ...,"### Key Rating Drivers – ALASKA AIR GROUP, INC...",### Key Rating Drivers\n\n- **Strong Market Po...,"### Key Rating Drivers – ALASKA AIR GROUP, INC...",### Key Rating Drivers\n\n- Consistent Insider...,### Key Rating Drivers\n\n- Historical Exposur...,"### Key Rating Drivers – ALASKA AIR GROUP, INC..."
5,"ALASKA AIR GROUP, INC.",2025,### Key Rating Drivers\n\n- **Resilient Revenu...,### Key Rating Drivers\n\n- Stabilizing and Im...,### Key Rating Drivers\n\n- Moderate Insider S...,### Key Rating Drivers\n\n- Strong Market Posi...,### Key Rating Drivers\n\n- Strong EBITDA Grow...,### Key Rating Drivers\n\n- **Strong Domestic ...,### Key Rating Drivers\n\n- Strong EBITDA Reco...,### Key Rating Drivers\n\n- Consistent Insider...,### Key Rating Drivers\n\n- Established Positi...,### Key Rating Drivers\n\n- Sustained EBITDA G...
6,"Merck & Co., Inc.",2022,### Key Rating Drivers\n\n- Resilient Revenue ...,### Key Rating Drivers\n\n- **Robust and Stabl...,### Key Rating Drivers\n\n- Steady Insider Act...,### Key Rating Drivers\n\n- Leading Market Pos...,### Key Rating Drivers\n\n- **Robust and Stabl...,### Key Rating Drivers\n\n- Solid Revenue and ...,### Key Rating Drivers\n\n- **Strong and Stabl...,### Key Rating Drivers\n\n- Routine Insider St...,### Key Rating Drivers\n\n- Strong Market Posi...,### Key Rating Drivers\n\n- **Strong and Stabl...
7,"Merck & Co., Inc.",2025,### Key Rating Drivers\n\n- Strong Market Posi...,### Key Rating Drivers\n\n- Strong and Stable ...,### Key Rating Drivers\n\n- Moderate Insider T...,### Key Rating Dri